# Trading Bot – Full Pipeline

**Anleitung:**
1. Accelerator auf **GPU T4 x2** stellen (Settings → Accelerator)
2. Dataset **busersteven/trading-raw-data** hinzufügen (Add data → Your datasets)
3. Alles ausführen: **Run All** oder **Save Version → Save & Run All (Commit)**

Die Pipeline:
- Lädt `scripts/kaggle_full_run.py` von GitHub (main); in der **Python-Zelle** steht `HORIZONS = [7]` (anpassbar; überschreibt den Stand von main)
- Klont danach das Repo für Trainings-/Backtest-Code
- Installiert Dependencies, kopiert Parquet-Daten
- Single-Horizon-Training (Walk-Forward) und Backtest
- Erzeugt Equity-Chart (`v2_7d_equity.png`) mit Benchmarks und Run-Manifest (`run_manifest.json`)
- Packt **alles** nach `/kaggle/working/kaggle_artifacts.tar.gz` inkl. Modell-Gewichte (.pt), Chart, Manifest, Log

In [ ]:
# Aktuellen Pipeline-Code von GitHub laden und ausführen
import os
import re
import subprocess
import sys
import time

# Single-Horizon (SCHRITT 20): VOR exec setzen — main() liest nur diese Variable.
HORIZONS = [7]
os.environ["KAGGLE_SH_HORIZONS"] = ",".join(str(h) for h in HORIZONS)
print(f"KAGGLE_SH_HORIZONS={os.environ['KAGGLE_SH_HORIZONS']} (wirkt in main())")

# Cache-Buster: raw.githubusercontent.com cached bis zu 5 Min
cache_bust = int(time.time())
url = f"https://raw.githubusercontent.com/stevenlangeshops/trading/main/scripts/kaggle_full_run.py?cb={cache_bust}"
r = subprocess.run(
    ["wget", "-q", "-O", "/kaggle/working/kaggle_full_run.py", url],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(r.stdout or "download ok")

path = "/kaggle/working/kaggle_full_run.py"
with open(path, encoding="utf-8") as f:
    code = f.read()

# Fallback: altes Skript mit fester Zeile SH_HORIZONS = [4, 7] (nur Ziffern in der Liste)
code, n_legacy = re.subn(
    r"^(\s*SH_HORIZONS\s*=\s*)\[\s*\d+\s*,\s*\d+\s*\]",
    lambda m: m.group(1) + repr(HORIZONS),
    code,
    flags=re.MULTILINE,
)
if n_legacy:
    print(f"[legacy] {n_legacy} SH_HORIZONS-Zeile(n) auf {HORIZONS!r} gesetzt")

# Modul-Cache leeren (verhindert Stale-Code bei Re-Runs)
for mod in list(sys.modules.keys()):
    if mod.startswith(("strategy", "models", "features")):
        del sys.modules[mod]

exec(code)